In [1]:
from __future__ import annotations

import json
import os
import platform
import sys
from datetime import datetime
from pathlib import Path
from typing import Any

import pandas as pd
import requests
from dotenv import load_dotenv
from requests import Response, Session
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

In [2]:
PROJECT_DIR = Path.cwd()

CONFIG_DIR = PROJECT_DIR / "config"
RAW_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
LOGS_DIR = PROJECT_DIR / "logs"

for directory in [
    CONFIG_DIR,
    RAW_DIR,
    PROCESSED_DIR,
    LOGS_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

print(f"Proyecto: {PROJECT_DIR}")
print(f"Respuestas crudas: {RAW_DIR}")
print(f"Resultados procesados: {PROCESSED_DIR}")

Proyecto: c:\Users\sacor\OneDrive\Documentos\solger
Respuestas crudas: c:\Users\sacor\OneDrive\Documentos\solger\data\raw
Resultados procesados: c:\Users\sacor\OneDrive\Documentos\solger\data\processed


In [3]:
env_path = PROJECT_DIR / ".env"

if not env_path.exists():
    env_template = """FUSIONSOLAR_BASE_URL=https://la5.fusionsolar.huawei.com
FUSIONSOLAR_USERNAME=REEMPLAZAR_USUARIO_NORTHBOUND
FUSIONSOLAR_PASSWORD=REEMPLAZAR_PASSWORD
FUSIONSOLAR_VERIFY_SSL=true
FUSIONSOLAR_TIMEOUT=60
"""
    
    env_path.write_text(env_template, encoding="utf-8")
    print(f"Archivo creado: {env_path}")
    print("Ábrelo y reemplaza el usuario y la contraseña.")
else:
    print(f"El archivo ya existe: {env_path}")

El archivo ya existe: c:\Users\sacor\OneDrive\Documentos\solger\.env


In [4]:
load_dotenv(PROJECT_DIR / ".env", override=True)

BASE_URL = os.getenv(
    "FUSIONSOLAR_BASE_URL",
    "https://la5.fusionsolar.huawei.com",
).rstrip("/")

USERNAME = os.getenv("FUSIONSOLAR_USERNAME", "").strip()
PASSWORD = os.getenv("FUSIONSOLAR_PASSWORD", "").strip()

VERIFY_SSL = (
    os.getenv("FUSIONSOLAR_VERIFY_SSL", "true")
    .strip()
    .lower()
    in {"true", "1", "yes", "si", "sí"}
)

TIMEOUT = int(os.getenv("FUSIONSOLAR_TIMEOUT", "60"))

configuration_errors = []

if not BASE_URL.startswith("https://"):
    configuration_errors.append(
        "FUSIONSOLAR_BASE_URL debe comenzar por https://"
    )

if not USERNAME or USERNAME == "REEMPLAZAR_USUARIO_NORTHBOUND":
    configuration_errors.append(
        "Debes configurar FUSIONSOLAR_USERNAME en el archivo .env"
    )

if not PASSWORD or PASSWORD == "REEMPLAZAR_PASSWORD":
    configuration_errors.append(
        "Debes configurar FUSIONSOLAR_PASSWORD en el archivo .env"
    )

if configuration_errors:
    raise ValueError(
        "Configuración incompleta:\n- "
        + "\n- ".join(configuration_errors)
    )

print("Configuración cargada correctamente")
print(f"Servidor: {BASE_URL}")
print(f"Usuario: {USERNAME[:3]}{'*' * max(len(USERNAME) - 3, 0)}")
print(f"Validar SSL: {VERIFY_SSL}")
print(f"Timeout: {TIMEOUT} segundos")

Configuración cargada correctamente
Servidor: https://la5.fusionsolar.huawei.com
Usuario: SOL***
Validar SSL: True
Timeout: 60 segundos


In [5]:
environment_info = {
    "fecha_ejecucion": datetime.now().isoformat(timespec="seconds"),
    "python": sys.version,
    "sistema_operativo": platform.platform(),
    "requests": requests.__version__,
    "base_url": BASE_URL,
    "verify_ssl": VERIFY_SSL,
}

display(pd.DataFrame(
    environment_info.items(),
    columns=["parametro", "valor"],
))

,parametro,valor
0,fecha_ejecucion,2026-08-05T22:36:54
1,python,"3.11.9 (tags/v3.11.9:de54cf5, Apr 2 2024, 10:..."
2,sistema_operativo,Windows-10-10.0.26200-SP0
3,requests,2.32.5
4,base_url,https://la5.fusionsolar.huawei.com
5,verify_ssl,True


In [6]:
def create_http_session() -> Session:
    """
    Crea una sesión HTTP para FusionSolar.

    Los reintentos se aplican únicamente a errores temporales del servidor.
    No se reintenta automáticamente el inicio de sesión para evitar bloqueos.
    """
    
    session = requests.Session()

    retry_strategy = Retry(
        total=3,
        connect=3,
        read=3,
        status=3,
        backoff_factor=1,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"],
        raise_on_status=False,
    )

    adapter = HTTPAdapter(
        max_retries=retry_strategy,
        pool_connections=10,
        pool_maxsize=10,
    )

    session.mount("https://", adapter)
    session.mount("http://", adapter)

    session.headers.update({
        "Accept": "application/json",
        "Content-Type": "application/json;charset=UTF-8",
        "User-Agent": "FusionSolar-Northbound-Explorer/0.1",
    })

    return session


session = create_http_session()

print("Sesión HTTP creada")

Sesión HTTP creada


In [7]:
def check_server(
    session: Session,
    base_url: str,
    timeout: int = 60,
) -> dict[str, Any]:
    """
    Comprueba si el dominio de FusionSolar es accesible.

    Esta prueba no valida todavía las credenciales Northbound.
    """
    
    result: dict[str, Any] = {
        "url": base_url,
        "reachable": False,
        "status_code": None,
        "content_type": None,
        "error": None,
    }

    try:
        response = session.get(
            base_url,
            timeout=timeout,
            verify=VERIFY_SSL,
            allow_redirects=True,
        )

        result.update({
            "reachable": True,
            "status_code": response.status_code,
            "final_url": response.url,
            "content_type": response.headers.get("Content-Type"),
        })

    except requests.exceptions.SSLError as exc:
        result["error"] = f"Error SSL: {exc}"

    except requests.exceptions.ConnectionError as exc:
        result["error"] = f"Error de conexión: {exc}"

    except requests.exceptions.Timeout:
        result["error"] = (
            f"El servidor no respondió en {timeout} segundos"
        )

    except requests.RequestException as exc:
        result["error"] = f"Error HTTP: {exc}"

    return result


server_check = check_server(
    session=session,
    base_url=BASE_URL,
    timeout=TIMEOUT,
)

display(pd.DataFrame(
    server_check.items(),
    columns=["prueba", "resultado"],
))

,prueba,resultado
0,url,https://la5.fusionsolar.huawei.com
1,reachable,True
2,status_code,200
3,content_type,text/html
4,error,None
5,final_url,https://la5.fusionsolar.huawei.com/pvmswebsite...


In [8]:
def safe_json(response: Response) -> dict[str, Any] | list[Any] | None:
    """
    Intenta convertir una respuesta HTTP a JSON.
    Retorna None cuando el contenido no es JSON válido.
    """
    
    try:
        return response.json()
    except ValueError:
        return None


def sanitize_headers(headers: dict[str, Any]) -> dict[str, Any]:
    """
    Oculta encabezados sensibles antes de imprimirlos o guardarlos.
    """
    
    sensitive_names = {
        "xsrf-token",
        "authorization",
        "cookie",
        "set-cookie",
    }

    sanitized = {}

    for key, value in headers.items():
        if key.lower() in sensitive_names:
            text = str(value)
            sanitized[key] = (
                f"{text[:6]}...OCULTO"
                if text
                else "OCULTO"
            )
        else:
            sanitized[key] = value

    return sanitized


def save_raw_json(
    data: Any,
    filename: str,
) -> Path:
    """
    Guarda una respuesta JSON conservando caracteres especiales.
    """
    
    output_path = RAW_DIR / filename

    with output_path.open(
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            data,
            file,
            ensure_ascii=False,
            indent=2,
            default=str,
        )

    return output_path


def inspect_response(
    response: Response,
    name: str,
    save_body: bool = True,
) -> dict[str, Any]:
    """
    Presenta información útil sin exponer el token completo.
    """
    
    body_json = safe_json(response)

    diagnostic = {
        "name": name,
        "request_method": response.request.method,
        "request_url": response.request.url,
        "status_code": response.status_code,
        "reason": response.reason,
        "elapsed_seconds": round(response.elapsed.total_seconds(), 3),
        "content_type": response.headers.get("Content-Type"),
        "response_headers": sanitize_headers(dict(response.headers)),
        "is_json": body_json is not None,
    }

    if save_body:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

        if body_json is not None:
            saved_path = save_raw_json(
                body_json,
                f"{timestamp}_{name}.json",
            )
        else:
            saved_path = RAW_DIR / f"{timestamp}_{name}.txt"
            saved_path.write_text(
                response.text,
                encoding="utf-8",
                errors="replace",
            )

        diagnostic["saved_path"] = str(saved_path)

    return diagnostic

In [9]:
{
  "userName": "usuario",
  "systemCode": "contraseña"
}

{'userName': 'usuario', 'systemCode': 'contraseña'}

In [10]:
def extract_xsrf_token(response: Response) -> str | None:
    """
    Busca el token en encabezados y cookies.

    Algunas versiones lo retornan directamente en XSRF-TOKEN;
    otras pueden conservarlo también como cookie.
    """
    
    possible_header_names = [
        "XSRF-TOKEN",
        "xsrf-token",
        "Xsrf-Token",
    ]

    for header_name in possible_header_names:
        token = response.headers.get(header_name)
        if token:
            return token

    possible_cookie_names = [
        "XSRF-TOKEN",
        "xsrf-token",
    ]

    for cookie_name in possible_cookie_names:
        token = response.cookies.get(cookie_name)
        if token:
            return token

        token = session.cookies.get(cookie_name)
        if token:
            return token

    return None


def login_northbound(
    session: Session,
    base_url: str,
    username: str,
    password: str,
    timeout: int = 60,
) -> tuple[Response, str | None]:
    """
    Inicia sesión en FusionSolar Northbound.

    No se implementan reintentos automáticos de autenticación para
    evitar bloquear el usuario por múltiples intentos fallidos.
    """
    
    url = f"{base_url}/thirdData/login"

    payload = {
        "userName": username,
        "systemCode": password,
    }

    response = session.post(
        url=url,
        json=payload,
        timeout=timeout,
        verify=VERIFY_SSL,
    )

    token = extract_xsrf_token(response)

    return response, token

In [11]:
login_response, xsrf_token = login_northbound(
    session=session,
    base_url=BASE_URL,
    username=USERNAME,
    password=PASSWORD,
    timeout=TIMEOUT,
)

login_diagnostic = inspect_response(
    response=login_response,
    name="login",
    save_body=True,
)

print("Diagnóstico del inicio de sesión")
print("-" * 60)

for key, value in login_diagnostic.items():
    if key != "response_headers":
        print(f"{key}: {value}")

print("-" * 60)

login_body = safe_json(login_response)

print("Cuerpo de la respuesta:")
print(json.dumps(
    login_body if login_body is not None else login_response.text[:1000],
    indent=2,
    ensure_ascii=False,
))

print("-" * 60)

if xsrf_token:
    print("XSRF-TOKEN encontrado correctamente")
    print(f"Vista parcial: {xsrf_token[:6]}...{xsrf_token[-4:]}")
else:
    print("No se encontró XSRF-TOKEN")
    print("No continúes todavía con las consultas de plantas.")

Diagnóstico del inicio de sesión
------------------------------------------------------------
name: login
request_method: POST
request_url: https://la5.fusionsolar.huawei.com/thirdData/login
status_code: 200
reason: OK
elapsed_seconds: 0.261
content_type: application/json;charset=UTF-8
is_json: True
saved_path: c:\Users\sacor\OneDrive\Documentos\solger\data\raw\20260805_223658_login.json
------------------------------------------------------------
Cuerpo de la respuesta:
{
  "data": null,
  "success": true,
  "failCode": 0,
  "params": {},
  "message": "Northbound API Reference: https://info.support.huawei.com/DpinfoAppBiz/#/u?i=K3T3uKIx&l=en_US"
}
------------------------------------------------------------
XSRF-TOKEN encontrado correctamente
Vista parcial: n-tiak...qppd


In [12]:
def validate_login(
    response: Response,
    token: str | None,
) -> None:
    """
    Detiene el notebook si la autenticación no parece correcta.
    """
    
    body = safe_json(response)

    if response.status_code >= 400:
        raise RuntimeError(
            f"El login respondió HTTP {response.status_code}. "
            f"Respuesta: {body or response.text[:500]}"
        )

    if not token:
        raise RuntimeError(
            "La respuesta no contiene XSRF-TOKEN. "
            "Debemos revisar el cuerpo y los encabezados antes de continuar."
        )

    print("Autenticación Northbound aparentemente correcta.")


validate_login(
    response=login_response,
    token=xsrf_token,
)

Autenticación Northbound aparentemente correcta.


In [13]:
session.headers.update({
    "XSRF-TOKEN": xsrf_token,
})

print("Token agregado a la sesión")
print("A partir de ahora las consultas usarán el mismo token")

Token agregado a la sesión
A partir de ahora las consultas usarán el mismo token


In [16]:
print("Encabezados activos de la sesión:")

for key, value in session.headers.items():
    if "token" in key.lower():
        print(f"{key}: {str(value)[:6]}...[OCULTO]")
    else:
        print(f"{key}: {value}")

Encabezados activos de la sesión:
User-Agent: FusionSolar-Northbound-Explorer/0.1
Accept-Encoding: gzip, deflate, br
Accept: application/json
Connection: keep-alive
Content-Type: application/json;charset=UTF-8
XSRF-TOKEN: n-tiak...[OCULTO]


In [19]:
from typing import Any
import requests

def post_northbound(
    session: requests.Session,
    endpoint: str,
    payload: dict[str, Any] | None = None,
    timeout: int = 60,
) -> requests.Response:
    """
    Realiza una petición POST autenticada a FusionSolar Northbound.
    """

    endpoint = endpoint if endpoint.startswith("/") else f"/{endpoint}"

    url = f"{BASE_URL}{endpoint}"

    response = session.post(
        url=url,
        json=payload or {},
        timeout=timeout,
        verify=VERIFY_SSL,
    )

    return response

In [20]:
response = post_northbound(
    session=session,
    endpoint="/thirdData/stations",
    payload={
        "pageNo": 1,
        "pageSize": 50,
    }
)

In [21]:
stations_response = post_northbound(
    session=session,
    endpoint="/thirdData/stations",
    payload={
        "pageNo": 1,
        "pageSize": 50,
    },
    timeout=TIMEOUT,
)

print("HTTP:", stations_response.status_code)

stations_body = stations_response.json()

print(json.dumps(
    stations_body,
    indent=2,
    ensure_ascii=False,
))

HTTP: 200
{
  "data": {
    "list": [
      {
        "capacity": 83.52,
        "contactMethod": null,
        "contactPerson": null,
        "gridConnectionDate": "2023-12-06T13:00:00-05:00",
        "latitude": "3.693611",
        "longitude": "-76.336389",
        "plantAddress": "ColombiaValle Del CaucaEl CerritoEl CerritoEl Cerrito",
        "plantCode": "NE=33795881",
        "plantName": "HDA LA CRUZ POZO 3"
      },
      {
        "capacity": 0.0,
        "contactMethod": "solger.colombia@gmail.com",
        "contactPerson": "solger.colombia@gmail.com",
        "gridConnectionDate": "2025-07-03T14:00:38-05:00",
        "latitude": "3.710442",
        "longitude": "-76.291416",
        "plantAddress": "Valle del CaucaEl CerritoEl Cerrito, Valle del Cauca, Colombia763527",
        "plantCode": "NE=37308996",
        "plantName": "SUPER GRANOS"
      },
      {
        "capacity": 145.2,
        "contactMethod": null,
        "contactPerson": null,
        "gridConnectionDate": 

In [23]:
if stations_body:
    df_stations = pd.json_normalize(stations_body)

    print("Dimensiones:", df_stations.shape)
    display(df_stations.head())
else:
    df_stations = pd.DataFrame()
    print("No se creó el DataFrame porque no se encontró la lista.")

Dimensiones: (1, 8)


,failCode,message,success,data.list,data.pageCount,data.pageNo,data.pageSize,data.total
0,0,get plant list success,True,"[{'capacity': 83.52, 'contactMethod': None, 'c...",1,1,100,16


In [24]:
stations_list = stations_body["data"]["list"]

df_stations = pd.DataFrame(stations_list)

print(f"Plantas encontradas: {len(df_stations)}")
display(df_stations)

Plantas encontradas: 16


,capacity,contactMethod,contactPerson,gridConnectionDate,latitude,longitude,plantAddress,plantCode,plantName
0,83.52,None,None,2023-12-06T13:00:00-05:00,3.693611,-76.336389,ColombiaValle Del CaucaEl CerritoEl CerritoEl ...,NE=33795881,HDA LA CRUZ POZO 3
1,0.00,solger.colombia@gmail.com,solger.colombia@gmail.com,2025-07-03T14:00:38-05:00,3.710442,-76.291416,"Valle del CaucaEl CerritoEl Cerrito, Valle del...",NE=37308996,SUPER GRANOS
2,145.20,None,None,2023-06-13T00:00:00-05:00,3.543076,-76.330311,"Valle del CaucaPalmiraPalmira, Valle del Cauca...",NE=33744070,SAN FELIPE - QUANTUM
3,0.00,solger.colombia@gmail.com,mail.com,2024-08-09T20:32:58-05:00,3.586866,-76.391021,"Valle del CaucaPalmiraPalmira, Valle del Cauca...",NE=33983774,BURRERA
4,0.00,solger.colombia@gmail.com,solger.colombia@gmail.com,2025-02-13T12:32:04-05:00,3.574051,-76.398471,ColombiaPalmira,NE=35502362,ANTOJO
5,83.52,solger.colombia@gmail.com,None,2023-10-23T00:00:00-05:00,3.692602,-76.298614,"Valle del CaucaEl CerritoEl Cerrito, Valle del...",NE=33777835,HDA LA CRUZ POZO 1
6,83.52,solger.colombia@gmail.com,None,2023-11-24T18:38:17-05:00,3.687233,-76.285932,"Valle del CaucaEl CerritoEl Cerrito, Valle del...",NE=33790681,HDA LA CRUZ POZO 2
7,0.00,solger.colombia@gmail.com,None,2024-04-13T16:37:21-05:00,3.761923,-76.327795,"Cl. 4 #261, Guacarí, Valle del Cauca, Colombia",NE=33879854,MOLINO SAN GERARDO
8,0.00,solger.colombia@gmail.com,solger.colombia@gmail.com,2024-09-11T18:16:36-05:00,3.570135,-76.417819,"Valle del CaucaPalmiraPalmira, Valle del Cauca...",NE=34035592,RIO GUACHAL
9,122.00,3182041320,None,2023-03-25T15:29:10-05:00,3.609018,-76.403360,"Palmira, Valle del Cauca, Colombia",NE=33731244,HDA PASO ANCHO


In [25]:
df_stations["capacity"] = pd.to_numeric(
    df_stations["capacity"],
    errors="coerce"
)

df_stations["latitude"] = pd.to_numeric(
    df_stations["latitude"],
    errors="coerce"
)

df_stations["longitude"] = pd.to_numeric(
    df_stations["longitude"],
    errors="coerce"
)

df_stations["gridConnectionDate"] = pd.to_datetime(
    df_stations["gridConnectionDate"],
    errors="coerce"
)

df_stations.dtypes

capacity                                float64
contactMethod                            object
contactPerson                            object
gridConnectionDate    datetime64[ns, UTC-05:00]
latitude                                float64
longitude                               float64
plantAddress                             object
plantCode                                object
plantName                                object
dtype: object

In [26]:
df_stations["plantId"] = (
    df_stations["plantCode"]
    .str.replace("NE=", "", regex=False)
)

display(
    df_stations[
        [
            "plantId",
            "plantCode",
            "plantName",
            "capacity"
        ]
    ]
)

,plantId,plantCode,plantName,capacity
0,33795881,NE=33795881,HDA LA CRUZ POZO 3,83.52
1,37308996,NE=37308996,SUPER GRANOS,0.00
2,33744070,NE=33744070,SAN FELIPE - QUANTUM,145.20
3,33983774,NE=33983774,BURRERA,0.00
4,35502362,NE=35502362,ANTOJO,0.00
5,33777835,NE=33777835,HDA LA CRUZ POZO 1,83.52
6,33790681,NE=33790681,HDA LA CRUZ POZO 2,83.52
7,33879854,NE=33879854,MOLINO SAN GERARDO,0.00
8,34035592,NE=34035592,RIO GUACHAL,0.00
9,33731244,NE=33731244,HDA PASO ANCHO,122.00


In [27]:
quality_summary = pd.DataFrame({
    "indicador": [
        "Número de plantas",
        "Capacidad mayor que cero",
        "Capacidad igual a cero",
        "Sin latitud",
        "Sin longitud",
        "Sin fecha de conexión",
        "Sin dirección",
    ],
    "cantidad": [
        len(df_stations),
        (df_stations["capacity"] > 0).sum(),
        (df_stations["capacity"] == 0).sum(),
        df_stations["latitude"].isna().sum(),
        df_stations["longitude"].isna().sum(),
        df_stations["gridConnectionDate"].isna().sum(),
        df_stations["plantAddress"].isna().sum(),
    ]
})

display(quality_summary)

,indicador,cantidad
0,Número de plantas,16
1,Capacidad mayor que cero,6
2,Capacidad igual a cero,10
3,Sin latitud,0
4,Sin longitud,0
5,Sin fecha de conexión,0
6,Sin dirección,0


In [28]:
df_capacity_zero = df_stations.loc[
    df_stations["capacity"].fillna(0).eq(0),
    [
        "plantCode",
        "plantName",
        "capacity",
        "gridConnectionDate"
    ]
].copy()

print(
    f"Plantas con capacidad cero: "
    f"{len(df_capacity_zero)} de {len(df_stations)}"
)

display(df_capacity_zero)

Plantas con capacidad cero: 10 de 16


,plantCode,plantName,capacity,gridConnectionDate
1,NE=37308996,SUPER GRANOS,0.0,2025-07-03 14:00:38-05:00
3,NE=33983774,BURRERA,0.0,2024-08-09 20:32:58-05:00
4,NE=35502362,ANTOJO,0.0,2025-02-13 12:32:04-05:00
7,NE=33879854,MOLINO SAN GERARDO,0.0,2024-04-13 16:37:21-05:00
8,NE=34035592,RIO GUACHAL,0.0,2024-09-11 18:16:36-05:00
10,NE=52329050,La unión,0.0,2026-02-12 00:00:00-05:00
11,NE=34081254,HERRAGRICOLA,0.0,2024-09-27 20:03:30-05:00
13,NE=35975392,PICHINDE,0.0,2025-04-29 15:10:16-05:00
14,NE=33945451,HDA MONICA,0.0,2024-06-27 13:23:17-05:00
15,NE=35899144,LA ESPERANZA 2,0.0,2025-04-16 07:37:11-05:00


In [29]:
portfolio_summary = {
    "plantas": len(df_stations),
    "plantas_con_capacidad": int(
        (df_stations["capacity"] > 0).sum()
    ),
    "plantas_sin_capacidad": int(
        (df_stations["capacity"] == 0).sum()
    ),
    "capacidad_total_configurada_kwp": float(
        df_stations["capacity"].sum()
    ),
    "fecha_conexion_mas_antigua": (
        df_stations["gridConnectionDate"].min()
    ),
    "fecha_conexion_mas_reciente": (
        df_stations["gridConnectionDate"].max()
    ),
}

display(
    pd.DataFrame(
        portfolio_summary.items(),
        columns=["indicador", "valor"]
    )
)

,indicador,valor
0,plantas,16
1,plantas_con_capacidad,6
2,plantas_sin_capacidad,10
3,capacidad_total_configurada_kwp,662.96
4,fecha_conexion_mas_antigua,2023-03-25 15:29:10-05:00
5,fecha_conexion_mas_reciente,2026-02-12 00:00:00-05:00


In [30]:
station_columns = [
    "plantId",
    "plantCode",
    "plantName",
    "capacity",
    "gridConnectionDate",
    "latitude",
    "longitude",
    "plantAddress",
]

df_station_master = (
    df_stations[station_columns]
    .sort_values("plantName")
    .reset_index(drop=True)
)

csv_path = PROCESSED_DIR / "maestro_plantas_fusionsolar.csv"
excel_path = PROCESSED_DIR / "maestro_plantas_fusionsolar.xlsx"

df_station_master.to_csv(
    csv_path,
    index=False,
    encoding="utf-8-sig"
)

df_station_master.to_excel(
    excel_path,
    index=False
)

print("CSV:", csv_path)
print("Excel:", excel_path)

display(df_station_master)

ValueError: Excel does not support datetimes with timezones. Please ensure that datetimes are timezone unaware before writing to Excel.

In [31]:
selected_plant = df_station_master.loc[
    df_station_master["plantName"].eq("HDA LA CRUZ POZO 3")
].iloc[0]

SELECTED_PLANT_CODE = selected_plant["plantCode"]
SELECTED_PLANT_NAME = selected_plant["plantName"]

print("Planta seleccionada:", SELECTED_PLANT_NAME)
print("Código:", SELECTED_PLANT_CODE)
print("Capacidad:", selected_plant["capacity"], "kWp")

Planta seleccionada: HDA LA CRUZ POZO 3
Código: NE=33795881
Capacidad: 83.52 kWp


In [32]:
selected_plant = df_station_master.loc[
    df_station_master["plantName"].eq("HDA LA CRUZ POZO 3")
].iloc[0]

SELECTED_PLANT_CODE = selected_plant["plantCode"]
SELECTED_PLANT_NAME = selected_plant["plantName"]

print("Planta seleccionada:", SELECTED_PLANT_NAME)
print("Código:", SELECTED_PLANT_CODE)
print("Capacidad:", selected_plant["capacity"], "kWp")

Planta seleccionada: HDA LA CRUZ POZO 3
Código: NE=33795881
Capacidad: 83.52 kWp


In [33]:
devices_response = post_northbound(
    session=session,
    endpoint="/thirdData/getDevList",
    payload={
        "stationCodes": SELECTED_PLANT_CODE
    },
    timeout=TIMEOUT,
)

print("HTTP:", devices_response.status_code)

devices_body = devices_response.json()

print(json.dumps(
    devices_body,
    indent=2,
    ensure_ascii=False,
))

HTTP: 200
{
  "data": [
    {
      "devDn": "NE=33795885",
      "devName": "INV2-40kW",
      "devTypeId": 1,
      "esnCode": "6T2269011084",
      "id": 1000000033795885,
      "invType": "SUN2000-40KTL-M3",
      "latitude": 3.693611,
      "longitude": -76.336389,
      "model": "SUN2000-40KTL-M3",
      "optimizerNumber": 0,
      "softwareVersion": "V100R001C20SPC127",
      "stationCode": "NE=33795881"
    },
    {
      "devDn": "NE=33795889",
      "devName": "Meter-1",
      "devTypeId": 47,
      "esnCode": null,
      "id": 1000000033795889,
      "invType": null,
      "latitude": 3.693611,
      "longitude": -76.336389,
      "model": null,
      "optimizerNumber": null,
      "softwareVersion": null,
      "stationCode": "NE=33795881"
    },
    {
      "devDn": "NE=33795884",
      "devName": "INV1-40kW",
      "devTypeId": 1,
      "esnCode": "6T2269011072",
      "id": 1000000033795884,
      "invType": "SUN2000-40KTL-M3",
      "latitude": 3.693611,
      "longitud

In [34]:
print("success:", devices_body.get("success"))
print("failCode:", devices_body.get("failCode"))
print("message:", devices_body.get("message"))

if devices_body.get("success") is not True:
    raise RuntimeError(
        "La consulta de dispositivos falló. "
        f"Respuesta: {devices_body}"
    )

print("Consulta de dispositivos exitosa.")

success: True
failCode: 0
message: None
Consulta de dispositivos exitosa.


In [35]:
def extract_devices_list(body: dict) -> list[dict]:
    """
    Extrae la lista de dispositivos de distintas estructuras
    posibles de respuesta de FusionSolar.
    """

    data = body.get("data")

    if isinstance(data, list):
        return data

    if isinstance(data, dict):
        possible_keys = [
            "list",
            "records",
            "devices",
            "devList",
        ]

        for key in possible_keys:
            candidate = data.get(key)

            if isinstance(candidate, list):
                return candidate

    return []


devices_list = extract_devices_list(devices_body)

print("Dispositivos encontrados:", len(devices_list))

Dispositivos encontrados: 4


In [36]:
if "devTypeId" in df_all_devices.columns:
    portfolio_device_summary = (
        df_all_devices
        .groupby("devTypeId", dropna=False)
        .agg(
            cantidad=("devTypeId", "size"),
            plantas=(
                "plantName",
                "nunique"
            ) if "plantName" in df_all_devices.columns
            else ("devTypeId", "size"),
        )
        .reset_index()
        .sort_values("cantidad", ascending=False)
    )

    display(portfolio_device_summary)

NameError: name 'df_all_devices' is not defined